### Code Methodology: ###

    - Main Data Folder: HEL1OS-Part3

    - Dataset Choice: HEL1OS (Dates: 15, 16 and 17 of July 2024) --> Justification: 16th of July 2024 saw a flare.
    - Workflow: Preprocessing --> Time series thresholding (flux v/s time) --> Hard X-ray burst detection --> Cross Validation (GOES)
    - ML aspect: Random Forest (trained on all three days) and CNN (trained ONLY on 15th July 2024). Followed by comparitive plot.
    - Animated Plot: Energy Spectrum Evolution during the X1.9 flare (matplotlib.animation.FuncAnimation)
    - Temporary Self-posed Question: "Does HEL1OS hard X-ray burst timing and duration agree with GOES soft X-ray flare timing for the same events, and does detection sensitivity hold for the weaker M-class events on 17 July as well as the major X1.9 event on 16 July?"

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table

# Key Directory
BASE_DIR = "HEL1OS-Part3"

# Subsequent Child Folders under Main Directory
DAY_TO_SHIFT_FOLDERS = {
    "HLS_20240715": ["HLS_20240715", "HLS_20240715_Shift1"],   
    "HLS_20240716": ["HLS_20240716", "HLS_20240716_Shift1"],   
    "HLS_20240717": ["HLS_20240717", "HLS_20240717_Shift1"],   
}

SUBFOLDER = "events"
EVENT_FILENAME = "evt.fits"

HDU_DETECTOR_MAP = {
    1: "cdte",
    2: "cdte",
    3: "czt",
    4: "czt",
}

def extract_single_file(filepath):
    #Note: Extraction of ALL data has been done from the "events.fits" file
    try:
        with fits.open(filepath) as hdul:
            frames = []
            for hdu_index, detector in HDU_DETECTOR_MAP.items():
                if hdu_index >= len(hdul):
                    print(f"  [WARN] HDU {hdu_index} not present in {filepath}")
                    continue

                hdu = hdul[hdu_index]
                if not (hasattr(hdu, "columns") and hdu.columns is not None):
                    print(f"  [WARN] HDU {hdu_index} has no table data in {filepath}")
                    continue

                table = Table(hdu.data)
                colnames = table.colnames

                if "mjd" not in colnames or "ener" not in colnames:         # 'mjd' --> Modified Julian Date
                    print(f"  [WARN] HDU {hdu_index} missing 'mjd' or 'ener': {colnames}")
                    continue

                df = pd.DataFrame({
                    "mjd": np.array(table["mjd"]).astype(float),
                    "energy": np.array(table["ener"]).astype(float),
                    "detector": detector,
                    "hdu": hdu_index,
                })

                if "utc-isot" in colnames:
                    df["utc_isot"] = np.array(table["utc-isot"]).astype(str)
                if "pix" in colnames:
                    df["pix"] = np.array(table["pix"])

                frames.append(df)

            if not frames:
                return None
            return pd.concat(frames, ignore_index=True)

    except Exception as e:
        print(f"  [ERROR] Could not process {filepath}: {e}")
        return None


def find_event_file(base_dir, shift_folder, subfolder, event_filename):     # Called subsequently in UDF extract_day()
    filepath = os.path.join(base_dir, shift_folder, subfolder, event_filename)
    if os.path.isfile(filepath):
        return filepath
    # fallback: glob for any FITS file if the exact name differs
    candidates = glob.glob(os.path.join(base_dir, shift_folder, subfolder, "*.fits"))
    return candidates[0] if candidates else None


def extract_day(base_dir, day_label, shift_folders, subfolder, event_filename):
    #Extracting and merging all shift-folders belonging to one logical day
    shift_frames = []

    for shift_folder in shift_folders:
        filepath = find_event_file(base_dir, shift_folder, subfolder, event_filename)
        if filepath is None:
            print(f"[{day_label}] No FITS file found in {shift_folder}/{subfolder}/ — skipping shift.")
            continue

        df = extract_single_file(filepath)
        if df is None or len(df) == 0:
            print(f"[{day_label}] No usable data extracted from {filepath}")
            continue

        df["shift_folder"] = shift_folder
        shift_frames.append(df)
        print(f"[{day_label}] Loaded {len(df)} rows from shift '{shift_folder}' "
              f"(cdte: {(df['detector']=='cdte').sum()}, czt: {(df['detector']=='czt').sum()})")

    if not shift_frames:
        print(f"[{day_label}] No data extracted from any shift.")
        return pd.DataFrame()

    day_df = pd.concat(shift_frames, ignore_index=True)
    day_df["day"] = day_label  # unified day label regardless of shift count
    print(f"[{day_label}] TOTAL merged rows: {len(day_df)} "
          f"(from {len(shift_frames)} shift file(s))")
    return day_df


def extract_all_days(base_dir, day_to_shift_folders, subfolder, event_filename):
    all_frames = []
    for day_label, shift_folders in day_to_shift_folders.items():
        day_df = extract_day(base_dir, day_label, shift_folders, subfolder, event_filename)
        if len(day_df) > 0:
            all_frames.append(day_df)

    if not all_frames:
        print("No data extracted across any day.")
        return pd.DataFrame(columns=["mjd", "energy", "detector", "hdu", "day"])

    combined = pd.concat(all_frames, ignore_index=True)
    combined = combined.sort_values("mjd").reset_index(drop=True)
    return combined


if __name__ == "__main__":
    combined_df = extract_all_days(BASE_DIR, DAY_TO_SHIFT_FOLDERS, SUBFOLDER, EVENT_FILENAME)

    print(f"\nTotal combined rows across all days: {len(combined_df)}")
    if len(combined_df) > 0:
        print(combined_df.head())
        print("\nDetector breakdown by day:")
        print(combined_df.groupby(["day", "detector"]).size())
        print("\nShift breakdown by day (confirms merge worked):")
        print(combined_df.groupby(["day", "shift_folder"]).size())

        out_path = "hel1os_combined_events.csv"
        combined_df.to_csv(out_path, index=False)
        print(f"\nSaved combined dataset to: {out_path}")

[HLS_20240715] Loaded 2501075 rows from shift 'HLS_20240715' (cdte: 71156, czt: 2429919)
[HLS_20240715] Loaded 2870010 rows from shift 'HLS_20240715_Shift1' (cdte: 389081, czt: 2480929)
[HLS_20240715] TOTAL merged rows: 5371085 (from 2 shift file(s))
[HLS_20240716] Loaded 8672443 rows from shift 'HLS_20240716' (cdte: 4834888, czt: 3837555)
[HLS_20240716] Loaded 2721150 rows from shift 'HLS_20240716_Shift1' (cdte: 264965, czt: 2456185)
[HLS_20240716] TOTAL merged rows: 11393593 (from 2 shift file(s))
[HLS_20240717] Loaded 2547829 rows from shift 'HLS_20240717' (cdte: 100798, czt: 2447031)
[HLS_20240717] Loaded 3434398 rows from shift 'HLS_20240717_Shift1' (cdte: 859089, czt: 2575309)
[HLS_20240717] TOTAL merged rows: 5982227 (from 2 shift file(s))

Total combined rows across all days: 22746905
            mjd      energy detector  hdu                 utc_isot    pix  \
0  60506.000132  196.859539      czt    4  2024-07-15T00:00:11.424  235.0   
1  60506.000132   36.286739     cdte    1 

In [2]:
import numpy as np
import pandas as pd
from astropy.time import Time

INPUT_CSV = "hel1os_combined_events.csv"        # Repository 1
OUTPUT_CSV = "hel1os_lightcurves.csv"           # Light curves data

BIN_SECONDS = 4.0          # light curve time resolution; 1-4s is typical for HXR flare timing
ENERGY_MIN = 8.0           # keV threshold, drop below CdTe's stated lower bound
ENERGY_MAX = 150.0         # keV threshold, drop above CZT's stated upper bound 

def load_and_clean(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} raw rows.")

    before = len(df)
    df = df[(df["energy"] >= ENERGY_MIN) & (df["energy"] <= ENERGY_MAX)]
    print(f"Energy filter [{ENERGY_MIN}-{ENERGY_MAX} keV]: kept {len(df)}/{before} rows "
          f"({before - len(df)} dropped)")

    # Convert MJD (Modified Julian Date) to seconds since start of its own day, for clean per-day binning
    df["mjd_day_floor"] = np.floor(df["mjd"])
    df["seconds_in_day"] = (df["mjd"] - df["mjd_day_floor"]) * 86400.0

    return df


def build_light_curve(df, bin_seconds):
    # Role: Events are binned according to the respective time interval mentioned
    bin_edges = np.arange(0, 86400 + bin_seconds, bin_seconds)
    df["time_bin"] = pd.cut(df["seconds_in_day"], bins=bin_edges, labels=bin_edges[:-1],
                             include_lowest=True)

    grouped = (
        df.groupby(["day", "detector", "time_bin"], observed=True)
        .size()
        .reset_index(name="counts")
    )
    grouped["time_bin"] = grouped["time_bin"].astype(float)
    grouped["count_rate"] = grouped["counts"] / bin_seconds  # counts/sec

    # Reconstruct an absolute MJD for each bin 
    day_map = df.groupby("day")["mjd_day_floor"].first().to_dict()
    grouped["mjd_bin"] = grouped.apply(
        lambda r: day_map[r["day"]] + r["time_bin"] / 86400.0, axis=1
    )

    return grouped.sort_values(["day", "detector", "time_bin"]).reset_index(drop=True)


if __name__ == "__main__":
    df = load_and_clean(INPUT_CSV)
    lc = build_light_curve(df, BIN_SECONDS)

    print(f"\nBuilt light curve: {len(lc)} time bins total")
    print(lc.groupby(["day", "detector"])["counts"].agg(["count", "mean", "max"]))

    lc.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved light curve data to: {OUTPUT_CSV}")

Loaded 22746905 raw rows.
Energy filter [8.0-150.0 keV]: kept 20535163/22746905 rows (2211742 dropped)

Built light curve: 85112 time bins total
                       count        mean    max
day          detector                          
HLS_20240715 cdte      13540   28.481093   1412
             czt       14686  306.360207   1447
HLS_20240716 cdte      13914  313.285037  28658
             czt       14893  394.450816  31382
HLS_20240717 cdte      13345   60.208992   3006
             czt       14734  313.100991  10405

Saved light curve data to: hel1os_lightcurves.csv


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_CSV = "hel1os_lightcurves.csv"
OUTPUT_CSV = "hel1os_detected_bursts_final.csv"

DETECTOR_CONFIG = {
    "cdte": {"rebin_factor": 4, "n_sigma": 8, "min_bins": 2, "min_peak_counts": 700},
    "czt":  {"rebin_factor": 1, "n_sigma": 5, "min_bins": 2, "min_peak_counts": 0},
}
BACKGROUND_PERCENTILE = 40


def rebin(group, factor):
    if factor == 1:
        return group.copy()
    g = group.sort_values("time_bin").reset_index(drop=True)
    g["rebin_group"] = g.index // factor
    out = g.groupby("rebin_group").agg(
        time_bin=("time_bin", "first"),
        counts=("counts", "sum"),
        day=("day", "first"),
        detector=("detector", "first"),
    ).reset_index(drop=True)
    return out


def estimate_background(counts, percentile):
    threshold_val = np.percentile(counts, percentile)
    quiet_subset = counts[counts <= threshold_val]
    return quiet_subset.mean(), quiet_subset.std()


def detect_bursts(group, n_sigma, min_bins, min_peak_counts=0, bg_percentile=BACKGROUND_PERCENTILE):
    counts = group["counts"].values
    bg_mean, bg_std = estimate_background(counts, bg_percentile)
    threshold = bg_mean + n_sigma * bg_std

    above = counts > threshold
    bursts = []
    i, n = 0, len(above)
    while i < n:
        if above[i]:
            j = i
            while j < n and above[j]:
                j += 1
            if (j - i) >= min_bins:
                seg = group.iloc[i:j]
                peak_idx = seg["counts"].idxmax()
                peak_val = group.loc[peak_idx, "counts"]
                if peak_val >= min_peak_counts:
                    bursts.append({
                        "day": group["day"].iloc[0],
                        "detector": group["detector"].iloc[0],
                        "start_time_bin": seg["time_bin"].iloc[0],
                        "end_time_bin": seg["time_bin"].iloc[-1],
                        "peak_time_bin": group.loc[peak_idx, "time_bin"],
                        "peak_counts": peak_val,
                        "background_mean": bg_mean,
                        "threshold": threshold,
                    })
            i = j
        else:
            i += 1
    return bursts, bg_mean, threshold


def hhmmss(seconds_in_day):
    h, rem = divmod(int(seconds_in_day), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


if __name__ == "__main__":
    lc = pd.read_csv(INPUT_CSV)

    all_bursts = []

    for day in sorted(lc["day"].unique()):
        fig, ax = plt.subplots(figsize=(14, 5))

        for detector, color in [("cdte", "tab:blue"), ("czt", "tab:orange")]:
            cfg = DETECTOR_CONFIG[detector]         # Retrieving dictionary values (d[key])
            group = lc[(lc["day"] == day) & (lc["detector"] == detector)].sort_values("time_bin")
            if len(group) == 0:
                continue

            group_plot = group  
            group_detect = rebin(group, cfg["rebin_factor"])  # detect at tuned resolution

            bursts, bg_mean, threshold = detect_bursts(
                group_detect, cfg["n_sigma"], cfg["min_bins"], cfg.get("min_peak_counts", 0)
            )
            all_bursts.extend(bursts)

            ax.plot(group_plot["time_bin"] / 3600.0, group_plot["counts"],          # Time Normalization (in relative hours)
                    label=f"{detector}", color=color, lw=0.6)
            ax.axhline(threshold, color=color, linestyle="--", alpha=0.4,
                        label=f"{detector} threshold ({cfg['n_sigma']}σ, rebin={cfg['rebin_factor']}x)")

            for b in bursts:
                ax.axvspan(b["start_time_bin"] / 3600.0, b["end_time_bin"] / 3600.0,
                            color=color, alpha=0.15)

            print(f"\n[{day} / {detector}] rebin={cfg['rebin_factor']}x, n_sigma={cfg['n_sigma']}, "
                  f"background={bg_mean:.1f}, threshold={threshold:.1f}, bursts found={len(bursts)}")
            for b in bursts:
                print(f"    {hhmmss(b['start_time_bin'])} - {hhmmss(b['end_time_bin'])} UTC, "
                      f"peak={b['peak_counts']:.0f} counts")

        ax.set_xlabel("Time (hours, UTC)")
        ax.set_ylabel("Counts per bin")
        ax.set_title(f"HEL1OS Light Curve — {day}")
        ax.legend(loc="upper right", fontsize=7)
        ax.set_yscale("log")
        ax.set_xlim(0, 24)
        fig.tight_layout()
        fig.savefig(f"light_curve_{day}.png", dpi=150)
        plt.close(fig)
        print(f"Saved plot: light_curve_{day}.png")
       

    bursts_df = pd.DataFrame(all_bursts)
    bursts_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nTotal bursts detected across all days/detectors: {len(bursts_df)}")
    print(f"Saved burst list to: {OUTPUT_CSV}")


[HLS_20240715 / cdte] rebin=4x, n_sigma=8, background=12.2, threshold=33.8, bursts found=3
    03:30:32 - 03:59:24 UTC, peak=1639 counts
    09:21:24 - 10:34:52 UTC, peak=4055 counts
    20:04:12 - 20:12:40 UTC, peak=1125 counts

[HLS_20240715 / czt] rebin=1x, n_sigma=5, background=193.6, threshold=685.6, bursts found=5
    09:24:52 - 09:25:00 UTC, peak=851 counts
    09:25:08 - 09:25:12 UTC, peak=779 counts
    09:26:32 - 09:26:36 UTC, peak=736 counts
    09:27:28 - 09:27:36 UTC, peak=972 counts
    09:28:48 - 09:28:56 UTC, peak=1447 counts
Saved plot: light_curve_HLS_20240715.png

[HLS_20240716 / cdte] rebin=4x, n_sigma=8, background=13.0, threshold=36.1, bursts found=6
    02:52:32 - 03:15:48 UTC, peak=2229 counts
    07:29:24 - 07:57:40 UTC, peak=4757 counts
    12:58:24 - 13:08:44 UTC, peak=702 counts
    13:11:00 - 14:28:56 UTC, peak=112948 counts
    21:11:00 - 21:37:56 UTC, peak=1069 counts
    21:45:36 - 22:25:00 UTC, peak=1907 counts

[HLS_20240716 / czt] rebin=1x, n_sigma=5

In [4]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# ---- CONFIGURE THIS ----
START_DATE = "2024-07-15"
END_DATE = "2024-07-18"   # Date extension to ensure reliability
API_KEY = "DEMO_KEY"      # Considered API  testKey

BURSTS_CSV = "hel1os_detected_bursts_final.csv"
MATCH_WINDOW_MINUTES = 20  # Proximity value set as a reference point for burst detection

def fetch_donki_flares(start_date, end_date, api_key):
    url = "https://api.nasa.gov/DONKI/FLR"
    params = {"startDate": start_date, "endDate": end_date, "api_key": api_key}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    rows = []
    for flare in data:
        rows.append({
            "flrID": flare.get("flrID"),
            "classType": flare.get("classType"),
            "beginTime": flare.get("beginTime"),
            "peakTime": flare.get("peakTime"),
            "endTime": flare.get("endTime"),
            "sourceLocation": flare.get("sourceLocation"),
            "activeRegionNum": flare.get("activeRegionNum"),
        })
    return pd.DataFrame(rows)


def day_folder_to_date(day_folder):
    # "HLS_20240715" -> "2024-07-15"
    digits = day_folder.replace("HLS_", "")
    return f"{digits[0:4]}-{digits[4:6]}-{digits[6:8]}"


def burst_peak_to_datetime(day_folder, peak_time_bin_seconds):
    date_str = day_folder_to_date(day_folder)
    base = datetime.strptime(date_str, "%Y-%m-%d")
    return base + timedelta(seconds=float(peak_time_bin_seconds))


def cross_validate(bursts_df, flares_df, window_minutes):
    flares_df = flares_df.copy()
    flares_df["peakTime_dt"] = pd.to_datetime(flares_df["peakTime"], utc=True).dt.tz_localize(None)

    results = []
    for _, b in bursts_df.iterrows():
        burst_dt = burst_peak_to_datetime(b["day"], b["peak_time_bin"])

        # find closest catalog flare in time
        if len(flares_df) > 0:
            diffs = (flares_df["peakTime_dt"] - burst_dt).abs()
            min_idx = diffs.idxmin()
            min_diff_minutes = diffs[min_idx].total_seconds() / 60.0
        else:
            min_idx = None
            min_diff_minutes = None

        matched = min_diff_minutes is not None and min_diff_minutes <= window_minutes

        results.append({
            "day": b["day"],
            "detector": b["detector"],
            "burst_peak_utc": burst_dt,
            "burst_peak_counts": b["peak_counts"],
            "matched_flare_class": flares_df.loc[min_idx, "classType"] if matched else None,
            "matched_flare_peak": flares_df.loc[min_idx, "peakTime"] if matched else None,
            "matched_active_region": flares_df.loc[min_idx, "activeRegionNum"] if matched else None,
            "time_diff_minutes": round(min_diff_minutes, 1) if min_diff_minutes is not None else None,
            "matched": matched,
        })

    return pd.DataFrame(results)


if __name__ == "__main__":
    print("Fetching DONKI flare catalog...")
    flares_df = fetch_donki_flares(START_DATE, END_DATE, API_KEY)
    print(f"\nCatalog flares found in window:")
    print(flares_df[["classType", "beginTime", "peakTime", "activeRegionNum"]]
          .to_string(index=False))

    bursts_df = pd.read_csv(BURSTS_CSV)
    validation_df = cross_validate(bursts_df, flares_df, MATCH_WINDOW_MINUTES)

    print(f"\n\n=== Validation Results (match window: {MATCH_WINDOW_MINUTES} min) ===")
    print(validation_df.to_string(index=False))

    n_matched = validation_df["matched"].sum()
    n_total = len(validation_df)
    print(f"\n{n_matched}/{n_total} detected bursts matched to a catalog flare within "
          f"{MATCH_WINDOW_MINUTES} minutes.")

    unmatched = validation_df[~validation_df["matched"]]
    if len(unmatched) > 0:
        print(f"\n{len(unmatched)} detected burst(s) had NO catalog match "
              f"(candidates for uncatalogued events, or noise if peak_counts is small):")
        print(unmatched[["day", "detector", "burst_peak_utc", "burst_peak_counts"]]
              .to_string(index=False))

    validation_df.to_csv("hel1os_validation_results.csv", index=False)
    print("\nSaved full validation table to: hel1os_validation_results.csv")

Fetching DONKI flare catalog...

Catalog flares found in window:
classType         beginTime          peakTime  activeRegionNum
     M1.2 2024-07-15T03:29Z 2024-07-15T03:45Z            13738
     M2.7 2024-07-15T09:21Z 2024-07-15T09:37Z            13738
     M1.9 2024-07-15T10:08Z 2024-07-15T10:13Z            13738
     M1.1 2024-07-15T10:36Z 2024-07-15T10:39Z            13738
     M1.4 2024-07-16T02:49Z 2024-07-16T03:01Z            13744
     M3.6 2024-07-16T07:24Z 2024-07-16T07:37Z            13753
     X1.9 2024-07-16T13:11Z 2024-07-16T13:26Z            13738
     M1.6 2024-07-16T21:06Z 2024-07-16T21:24Z            13753
     M1.9 2024-07-16T21:46Z 2024-07-16T22:06Z            13744
     M5.0 2024-07-17T06:26Z 2024-07-17T06:39Z            13743
     M3.4 2024-07-17T07:01Z 2024-07-17T07:08Z            13743
     M1.2 2024-07-17T09:26Z 2024-07-17T09:45Z            13738
     M1.2 2024-07-17T19:40Z 2024-07-17T19:51Z            13753
     M2.0 2024-07-17T19:53Z 2024-07-17T19:58Z        

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, jaccard_score

LIGHTCURVE_CSV = "hel1os_lightcurves.csv"
BURSTS_CSV = "hel1os_detected_bursts_final.csv"
VALIDATION_CSV = "hel1os_validation_results.csv"

TRAIN_DAYS = ["HLS_20240715", "HLS_20240716"]
TEST_DAYS = ["HLS_20240717"]

DETECTOR = "cdte"   
ROLL_WINDOW = 5      


def load_labeled_lightcurve():
    lc = pd.read_csv(LIGHTCURVE_CSV)
    lc = lc[lc["detector"] == DETECTOR].sort_values(["day", "time_bin"]).reset_index(drop=True)

    bursts = pd.read_csv(BURSTS_CSV)
    bursts = bursts[bursts["detector"] == DETECTOR]

    # Considering catalog-matched values as ground truth
    validation = pd.read_csv(VALIDATION_CSV)
    matched = validation[(validation["detector"] == DETECTOR) & (validation["matched"] == True)]

    # Building a lookup of (day, start, end) to assign labels 
    bursts["peak_time_bin_round"] = bursts["peak_time_bin"].round(0)

    lc["label"] = 0
    for _, b in bursts.iterrows():
        mask = (
            (lc["day"] == b["day"]) &
            (lc["time_bin"] >= b["start_time_bin"]) &
            (lc["time_bin"] <= b["end_time_bin"])
        )
        lc.loc[mask, "label"] = 1

    return lc


def engineer_features(lc):
    lc = lc.copy()
    lc = lc.sort_values(["day", "time_bin"]).reset_index(drop=True)

    feats = []
    for day, g in lc.groupby("day"):
        g = g.sort_values("time_bin").copy()
        g["roll_mean"] = g["counts"].rolling(ROLL_WINDOW, min_periods=1, center=True).mean()
        g["roll_std"] = g["counts"].rolling(ROLL_WINDOW, min_periods=1, center=True).std().fillna(0)
        g["gradient"] = g["counts"].diff().fillna(0)
        g["gradient2"] = g["gradient"].diff().fillna(0)  # curvature
        g["ratio_to_rollmean"] = g["counts"] / (g["roll_mean"] + 1e-6)
        feats.append(g)

    return pd.concat(feats, ignore_index=True)


if __name__ == "__main__":
    lc = load_labeled_lightcurve()
    lc = engineer_features(lc)

    print(f"Loaded {len(lc)} bins for detector={DETECTOR}. "
          f"Positive (burst) bins: {lc['label'].sum()} ({100*lc['label'].mean():.2f}%)")

    feature_cols = ["counts", "roll_mean", "roll_std", "gradient", "gradient2", "ratio_to_rollmean"]

    train = lc[lc["day"].isin(TRAIN_DAYS)]
    test = lc[lc["day"].isin(TEST_DAYS)]

    X_train, y_train = train[feature_cols], train["label"]
    X_test, y_test = test[feature_cols], test["label"]

    print(f"\nTrain: {len(X_train)} bins ({y_train.sum()} positive) from {TRAIN_DAYS}")
    print(f"Test:  {len(X_test)} bins ({y_test.sum()} positive) from {TEST_DAYS}")

    clf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced",
                                  random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    print("\n=== Classification Report (test = 17 July) ===")
    print(classification_report(y_test, y_pred, target_names=["quiet", "burst"]))

    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_test, y_pred))

    iou = jaccard_score(y_test, y_pred)
    print(f"\nIoU (Jaccard score) vs threshold-based ground truth: {iou:.3f}")

    importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("\nFeature importances:")
    print(importances)

    # Save predictions for later comparison/plotting against threshold + CNN methods
    test_out = test.copy()
    test_out["rf_prediction"] = y_pred
    test_out.to_csv(f"hel1os_rf_predictions_{DETECTOR}.csv", index=False)
    print(f"\nSaved predictions to: hel1os_rf_predictions_{DETECTOR}.csv")

Loaded 40799 bins for detector=cdte. Positive (burst) bins: 6286 (15.41%)

Train: 27454 bins (3965 positive) from ['HLS_20240715', 'HLS_20240716']
Test:  13345 bins (2321 positive) from ['HLS_20240717']

=== Classification Report (test = 17 July) ===
              precision    recall  f1-score   support

       quiet       0.98      0.97      0.98     11024
       burst       0.87      0.92      0.89      2321

    accuracy                           0.96     13345
   macro avg       0.92      0.94      0.93     13345
weighted avg       0.96      0.96      0.96     13345

Confusion matrix [[TN, FP], [FN, TP]]:
[[10701   323]
 [  197  2124]]

IoU (Jaccard score) vs threshold-based ground truth: 0.803

Feature importances:
roll_mean            0.414638
roll_std             0.243655
counts               0.239159
gradient2            0.060995
gradient             0.032122
ratio_to_rollmean    0.009430
dtype: float64

Saved predictions to: hel1os_rf_predictions_cdte.csv


In [6]:
import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
except ImportError:
    raise SystemExit(
        "PyTorch is not installed"
    )

LIGHTCURVE_CSV = "hel1os_lightcurves.csv"
BURSTS_CSV = "hel1os_detected_bursts_final.csv"

TRAIN_DAYS = ["HLS_20240715", "HLS_20240716"]
TEST_DAYS = ["HLS_20240717"]
DETECTOR = "cdte"   

WINDOW_SIZE = 32     
STRIDE = 4           
EPOCHS = 30
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
ANOMALY_PERCENTILE = 95  # Windows above this hardcoded reconstruction-error percentile
                          

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")           # "cpu" was taken as the default value


def load_labeled_lightcurve():
    lc = pd.read_csv(LIGHTCURVE_CSV)
    lc = lc[lc["detector"] == DETECTOR].sort_values(["day", "time_bin"]).reset_index(drop=True)

    bursts = pd.read_csv(BURSTS_CSV)
    bursts = bursts[bursts["detector"] == DETECTOR]

    lc["label"] = 0
    for _, b in bursts.iterrows():
        mask = (
            (lc["day"] == b["day"]) &
            (lc["time_bin"] >= b["start_time_bin"]) &
            (lc["time_bin"] <= b["end_time_bin"])
        )
        lc.loc[mask, "label"] = 1

    return lc


def make_windows(counts, labels, window_size, stride):
    n = len(counts)
    windows, window_labels, start_idxs = [], [], []
    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        windows.append(counts[start:end])
        window_labels.append(int(labels[start:end].max()))
        start_idxs.append(start)
    return np.array(windows, dtype=np.float32), np.array(window_labels), np.array(start_idxs)


def normalize(windows, mean, std):
    return (windows - mean) / (std + 1e-6)


class ConvAutoencoder(nn.Module):
    # Architecture: Conv1D(16)->Pool->Conv1D(8)->Pool ->Conv1D(8)->Upsample->Conv1D(16)->Upsample->Conv1D(1)

    def __init__(self, window_size):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2, ceil_mode=True),                      
            nn.Conv1d(16, 8, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2, ceil_mode=True),                      
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(8, 8, kernel_size=5, padding=2), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv1d(8, 16, kernel_size=5, padding=2), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv1d(16, 1, kernel_size=5, padding=2),            # linear output
        )
        self.window_size = window_size

    def forward(self, x):
        # x: (batch, 1, window_size)
        z = self.encoder(x)
        out = self.decoder(z)
        if out.shape[-1] != x.shape[-1]:
            out = out[..., :x.shape[-1]] if out.shape[-1] > x.shape[-1] else \
                  nn.functional.pad(out, (0, x.shape[-1] - out.shape[-1]))
        return out


def train_autoencoder(model, train_tensor, epochs, batch_size, lr):
    dataset = TensorDataset(train_tensor, train_tensor)
    n_val = max(1, int(0.1 * len(dataset)))
    n_train = len(dataset) - n_val
    train_set, val_set = torch.utils.data.random_split(dataset, [n_train, n_val])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= n_train

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                val_loss += criterion(out, yb).item() * xb.size(0)
        val_loss /= n_val

        print(f"Epoch {epoch+1}/{epochs} - train_loss: {train_loss:.5f} - val_loss: {val_loss:.5f}")


def reconstruction_errors(model, data_tensor, batch_size=256):
    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(0, len(data_tensor), batch_size):
            batch = data_tensor[i:i+batch_size].to(DEVICE)
            recon = model(batch)
            err = torch.mean((batch - recon) ** 2, dim=(1, 2))
            errors.append(err.cpu().numpy())
    return np.concatenate(errors)


if __name__ == "__main__":
    print(f"Using device: {DEVICE}")

    lc = load_labeled_lightcurve()

    train_lc = lc[lc["day"].isin(TRAIN_DAYS)].sort_values("time_bin")
    test_lc = lc[lc["day"].isin(TEST_DAYS)].sort_values("time_bin")

    train_windows, train_labels, _ = make_windows(
        train_lc["counts"].values, train_lc["label"].values, WINDOW_SIZE, STRIDE
    )
    test_windows, test_labels, test_starts = make_windows(
        test_lc["counts"].values, test_lc["label"].values, WINDOW_SIZE, STRIDE
    )

    quiet_train_windows = train_windows[train_labels == 0]
    print(f"Training windows (quiet only): {len(quiet_train_windows)} / {len(train_windows)} total")
    print(f"Test windows (quiet+burst): {len(test_windows)} "
          f"({test_labels.sum()} contain a burst)")

    mean, std = quiet_train_windows.mean(), quiet_train_windows.std()
    quiet_train_norm = normalize(quiet_train_windows, mean, std)
    test_norm = normalize(test_windows, mean, std)

    # Reshaping to (N, channels=1, window_size) for PyTorch Conv1d
    train_tensor = torch.tensor(quiet_train_norm, dtype=torch.float32).unsqueeze(1)
    test_tensor = torch.tensor(test_norm, dtype=torch.float32).unsqueeze(1)

    model = ConvAutoencoder(WINDOW_SIZE).to(DEVICE)
    train_autoencoder(model, train_tensor, EPOCHS, BATCH_SIZE, LEARNING_RATE)

    train_errors = reconstruction_errors(model, train_tensor)
    threshold = np.percentile(train_errors, ANOMALY_PERCENTILE)
    print(f"\nAnomaly threshold (train quiet-window reconstruction error, "
          f"{ANOMALY_PERCENTILE}th percentile): {threshold:.5f}")

    test_errors = reconstruction_errors(model, test_tensor)
    predicted_anomaly = (test_errors > threshold).astype(int)

    from sklearn.metrics import classification_report, confusion_matrix, jaccard_score

    print("\n=== CNN Autoencoder Anomaly Detection Report (test = 17 July) ===")
    print(classification_report(test_labels, predicted_anomaly, target_names=["quiet", "burst"]))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(test_labels, predicted_anomaly))
    print(f"IoU (Jaccard score): {jaccard_score(test_labels, predicted_anomaly):.3f}")

    out = pd.DataFrame({
        "window_start_idx": test_starts,
        "true_label": test_labels,
        "reconstruction_error": test_errors,
        "cnn_predicted_anomaly": predicted_anomaly,
    })
    out.to_csv(f"hel1os_cnn_predictions_{DETECTOR}.csv", index=False)
    print(f"\nSaved predictions to: hel1os_cnn_predictions_{DETECTOR}.csv")

Using device: cpu
Training windows (quiet only): 5074 / 6856 total
Test windows (quiet+burst): 3329 (619 contain a burst)
Epoch 1/30 - train_loss: 0.71724 - val_loss: 0.57230
Epoch 2/30 - train_loss: 0.51923 - val_loss: 0.52145
Epoch 3/30 - train_loss: 0.49532 - val_loss: 0.50224
Epoch 4/30 - train_loss: 0.48035 - val_loss: 0.48663
Epoch 5/30 - train_loss: 0.47430 - val_loss: 0.48531
Epoch 6/30 - train_loss: 0.46232 - val_loss: 0.46949
Epoch 7/30 - train_loss: 0.45220 - val_loss: 0.45995
Epoch 8/30 - train_loss: 0.44177 - val_loss: 0.44707
Epoch 9/30 - train_loss: 0.42508 - val_loss: 0.42633
Epoch 10/30 - train_loss: 0.39820 - val_loss: 0.39314
Epoch 11/30 - train_loss: 0.36391 - val_loss: 0.37359
Epoch 12/30 - train_loss: 0.34344 - val_loss: 0.33748
Epoch 13/30 - train_loss: 0.31230 - val_loss: 0.30874
Epoch 14/30 - train_loss: 0.28870 - val_loss: 0.29182
Epoch 15/30 - train_loss: 0.26938 - val_loss: 0.27882
Epoch 16/30 - train_loss: 0.25516 - val_loss: 0.26145
Epoch 17/30 - train_los

In [7]:
# Comparison plot between ML models (Random Forest vs CNN Autoencoder)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LIGHTCURVE_CSV = "hel1os_lightcurves.csv"
BURSTS_CSV = "hel1os_detected_bursts_final.csv"
RF_CSV = "hel1os_rf_predictions_cdte.csv"
CNN_CSV = "hel1os_cnn_predictions_cdte.csv"

DETECTOR = "cdte"
DAY = "HLS_20240717"

WINDOW_SIZE = 32   
STRIDE = 4         


def bins_to_spans(time_bins_seconds, flags):
    spans = []
    n = len(flags)
    i = 0

    if n > 1:
        diffs = np.diff(time_bins_seconds)
        bin_width = np.median(diffs[diffs > 0]) if np.any(diffs > 0) else 4.0
    else:
        bin_width = 4.0

    while i < n:
        if flags[i]:
            j = i
            while j < n and flags[j]:
                j += 1
            start_s = time_bins_seconds[i]
            end_s = time_bins_seconds[j - 1] + bin_width
            spans.append((start_s / 3600.0, end_s / 3600.0))
            i = j
        else:
            i += 1
    return spans


def load_light_curve():
    lc = pd.read_csv(LIGHTCURVE_CSV)
    lc = lc[(lc["day"] == DAY) & (lc["detector"] == DETECTOR)].sort_values("time_bin").reset_index(drop=True)
    return lc


def threshold_spans():
    bursts = pd.read_csv(BURSTS_CSV)
    bursts = bursts[(bursts["day"] == DAY) & (bursts["detector"] == DETECTOR)]
    return [(row["start_time_bin"] / 3600.0, row["end_time_bin"] / 3600.0)
            for _, row in bursts.iterrows()]


def rf_spans():
    rf = pd.read_csv(RF_CSV)
    rf = rf[rf["day"] == DAY].sort_values("time_bin").reset_index(drop=True)
    return bins_to_spans(rf["time_bin"].values, rf["rf_prediction"].values)


def cnn_spans(lc):
    cnn = pd.read_csv(CNN_CSV)
    time_bin_values = lc["time_bin"].values  # same order Step 7 used

    flagged_bin_mask = np.zeros(len(lc), dtype=bool)
    for _, row in cnn[cnn["cnn_predicted_anomaly"] == 1].iterrows():
        start_idx = int(row["window_start_idx"])
        end_idx = min(start_idx + WINDOW_SIZE, len(lc))
        flagged_bin_mask[start_idx:end_idx] = True

    return bins_to_spans(time_bin_values, flagged_bin_mask.astype(int))


if __name__ == "__main__":
    lc = load_light_curve()

    th_spans = threshold_spans()
    rf_sp = rf_spans()
    cnn_sp = cnn_spans(lc)

    print(f"Threshold spans: {len(th_spans)}")
    print(f"Random Forest spans: {len(rf_sp)}")
    print(f"CNN spans: {len(cnn_sp)}")

    fig, (ax_lc, ax_tracks) = plt.subplots(
        2, 1, figsize=(15, 7), sharex=True,
        gridspec_kw={"height_ratios": [2.5, 1]}
    )

    
    ax_lc.plot(lc["time_bin"] / 3600.0, lc["counts"], color="black", lw=0.6)
    ax_lc.set_ylabel("Counts per bin")
    ax_lc.set_yscale("log")
    ax_lc.set_title(f"HEL1OS Light Curve & Detection Method Comparison — {DAY} ({DETECTOR})")

   
    track_specs = [
        ("Threshold\n(ground truth)", th_spans, "tab:red", 2),
        ("Random Forest", rf_sp, "tab:green", 1),
        ("CNN Autoencoder", cnn_sp, "tab:purple", 0),
    ]

    for label, spans, color, y in track_specs:
        bars = [(start, end - start) for start, end in spans]
        ax_tracks.broken_barh(bars, (y - 0.35, 0.7), facecolors=color, alpha=0.8)

    ax_tracks.set_yticks([0, 1, 2])
    ax_tracks.set_yticklabels(["CNN Autoencoder", "Random Forest", "Threshold\n(ground truth)"])
    ax_tracks.set_xlabel("Time (hours, UTC)")
    ax_tracks.set_xlim(0, 24)
    ax_tracks.set_ylim(-0.6, 2.6)
    ax_tracks.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    fig.savefig(f"comparison_plot_{DAY}_{DETECTOR}.png", dpi=150)
    plt.close(fig)
    print(f"\nSaved comparison plot: comparison_plot_{DAY}_{DETECTOR}.png")

Threshold spans: 5
Random Forest spans: 54
CNN spans: 9

Saved comparison plot: comparison_plot_HLS_20240717_cdte.png


In [8]:
# Creative Element: Animation spectrum of X1.9 flare event (16-July-2024)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

INPUT_CSV = "hel1os_combined_events.csv"
OUTPUT_GIF = "xray_spectrum_evolution_X1.9.gif"

DAY = "HLS_20240716"
DETECTOR = "cdte"          

# Time window around the X1.9 event (UTC seconds-in-day).
WINDOW_START_SEC = 12 * 3600 + 50 * 60   
WINDOW_END_SEC = 14 * 3600 + 40 * 60     

N_FRAMES = 20               
ENERGY_MIN, ENERGY_MAX = 8.0, 70.0
N_ENERGY_BINS = 30

FPS = 2                     # Playback speed of the saved GIF

# Overall title shown at the top of the GIF, above the per-frame subtitle.
GIF_TITLE = "HEL1OS Observation of the X1.9 Solar Flare — 16 July 2024"


def load_window_events():
    df = pd.read_csv(INPUT_CSV)
    df = df[(df["day"] == DAY) & (df["detector"] == DETECTOR)].copy()

    df["mjd_day_floor"] = np.floor(df["mjd"])
    df["seconds_in_day"] = (df["mjd"] - df["mjd_day_floor"]) * 86400.0

    df = df[(df["seconds_in_day"] >= WINDOW_START_SEC) &
            (df["seconds_in_day"] <= WINDOW_END_SEC)]
    df = df[(df["energy"] >= ENERGY_MIN) & (df["energy"] <= ENERGY_MAX)]

    print(f"Loaded {len(df)} events in the {WINDOW_START_SEC/3600:.2f}-"
          f"{WINDOW_END_SEC/3600:.2f}h UTC window for {DETECTOR}.")
    return df


def hhmmss(seconds_in_day):
    h, rem = divmod(int(seconds_in_day), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


if __name__ == "__main__":
    df = load_window_events()

    frame_edges = np.linspace(WINDOW_START_SEC, WINDOW_END_SEC, N_FRAMES + 1)
    energy_bins = np.linspace(ENERGY_MIN, ENERGY_MAX, N_ENERGY_BINS + 1)

    # Precompute histogram + hardness ratio (high/low energy counts) per frame
    frame_hists = []
    hardness_ratios = []
    frame_counts = []
    for i in range(N_FRAMES):
        t0, t1 = frame_edges[i], frame_edges[i + 1]
        seg = df[(df["seconds_in_day"] >= t0) & (df["seconds_in_day"] < t1)]
        hist, _ = np.histogram(seg["energy"], bins=energy_bins)
        frame_hists.append(hist)
        frame_counts.append(len(seg))

        low = (seg["energy"] < 25).sum()
        high = (seg["energy"] >= 25).sum()
        hardness_ratios.append(high / low if low > 0 else np.nan)

    max_count = max(h.max() for h in frame_hists) if frame_hists else 1

    fig, (ax_spec, ax_hard) = plt.subplots(
        2, 1, figsize=(9, 7.5), gridspec_kw={"height_ratios": [2.5, 1]}
    )

    # Figure-level title for the GIF as a whole.
    fig.suptitle(GIF_TITLE, fontsize=13, fontweight="bold")

    bin_centers = 0.5 * (energy_bins[:-1] + energy_bins[1:])
    bin_width = energy_bins[1] - energy_bins[0]
    bars = ax_spec.bar(bin_centers, frame_hists[0], width=bin_width * 0.9, color="tab:orange")
    ax_spec.set_xlim(ENERGY_MIN, ENERGY_MAX)
    ax_spec.set_yscale("log")
    ax_spec.set_ylim(0.5, max_count * 1.5) 
    ax_spec.set_xlabel("Energy (keV)")
    ax_spec.set_ylabel("Counts per bin")
    title = ax_spec.set_title("")

    frame_times_hr = [(frame_edges[i] + frame_edges[i+1]) / 2 / 3600.0 for i in range(N_FRAMES)]
    ax_hard.plot(frame_times_hr, hardness_ratios, color="tab:blue", marker="o", markersize=3)
    ax_hard.set_xlabel("Time (hours, UTC)")
    ax_hard.set_ylabel("Hardness ratio\n(>25 keV / <25 keV)")
    marker, = ax_hard.plot([], [], "o", color="red", markersize=10, zorder=5)

    # Leave room at the top for the suptitle.
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    def update(frame_idx):
        hist = frame_hists[frame_idx]
        for bar, h in zip(bars, hist):
            bar.set_height(h)

        t0, t1 = frame_edges[frame_idx], frame_edges[frame_idx + 1]
        title.set_text(
            f"HEL1OS CdTe Energy Spectrum — {DAY[4:8]}/{DAY[8:10]} UTC\n"
            f"{hhmmss(t0)} - {hhmmss(t1)} UTC ({frame_counts[frame_idx]} counts)"
        )

        marker.set_data([frame_times_hr[frame_idx]], [hardness_ratios[frame_idx]])
        return list(bars) + [title, marker]

    anim = animation.FuncAnimation(
        fig, update, frames=N_FRAMES, interval=1000 / FPS, blit=False, repeat=True
    )

    writer = animation.PillowWriter(fps=FPS)
    anim.save(OUTPUT_GIF, writer=writer)
    plt.close(fig)

    print(f"\nSaved animation: {OUTPUT_GIF}")
    print(f"{N_FRAMES} frames, {WINDOW_START_SEC/3600:.2f}h to {WINDOW_END_SEC/3600:.2f}h UTC, "
          f"~{(WINDOW_END_SEC-WINDOW_START_SEC)/N_FRAMES:.0f}s per frame")

Loaded 3999115 events in the 12.83-14.67h UTC window for cdte.

Saved animation: xray_spectrum_evolution_X1.9.gif
20 frames, 12.83h to 14.67h UTC, ~330s per frame
